# bn-weight-bias-init-pattern — faded example 1: Complete the BN init function body

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `bn-weight-bias-init-pattern`. The last cell reports your progress on the `GAN: BN weight=1 bias=0 init` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BN weight=1 bias=0 init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bn-weight-bias-init-pattern`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bn-weight-bias-init-pattern"
DD_SUBTOPIC = "GAN: BN weight=1 bias=0 init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

DCGAN sets every BatchNorm layer's gamma to a draw from `N(1, 0.02)` and its beta to exactly zero, while leaving all other module types alone. The recursive `model.apply(init_fn)` requires `init_fn` to guard on the BN type itself.

## Faded exercise 1

### Faded — complete the BN init function body

Implement `apply_bn_init(model)`. The outer structure (the guard and the `model.apply` call) is given. You must fill the body that runs *only for BatchNorm modules*: resample gamma from `N(1, 0.02)` and zero beta. Do not touch non-BN layers.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch.nn as nn

t.manual_seed(0)

def apply_bn_init(model: nn.Module) -> nn.Module:
    def init_fn(m):
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.zeros_(m.bias)
    model.apply(init_fn)
    return model

model = nn.Sequential(nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8), nn.Linear(8, 4))
apply_bn_init(model)


def _test():
    import torch.nn as nn
    bn = model[1]
    assert isinstance(bn, nn.BatchNorm2d)
    # beta exactly zero
    assert t.allclose(bn.bias, t.zeros_like(bn.bias)), "BN bias must be all zeros"
    # gamma sampled near mean 1.0 (8 channels -> loose tolerance)
    assert abs(bn.weight.mean().item() - 1.0) < 0.1, "BN gamma mean should be ~1.0"
    assert bn.weight.std().item() < 0.2, "BN gamma spread should be small (~0.02)"
    # gamma is not the constant default of 1.0 (init_ actually ran)
    assert not t.allclose(bn.weight, t.ones_like(bn.weight)), "gamma must be resampled, not left at 1"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn

t.manual_seed(0)

def apply_bn_init(model: nn.Module) -> nn.Module:
    def init_fn(m):
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.zeros_(m.bias)
    model.apply(init_fn)
    return model

model = nn.Sequential(nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8), nn.Linear(8, 4))
apply_bn_init(model)
```
</details>